# Baseline Comparison

This notebook compares M-Core, B1, B2, and B3 baselines side-by-side.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

In [ ]:
# Define which baselines to compare (in order: M-Core, B1, B2, B3)
BASELINES = {
    'M-Core': 'results_mcore.json',
    'B2 (Sequence)': 'results_b2_sequence.json',
    # Add more as you generate them:
    # 'B1 (Schema)': 'results_b1_schema.json',
    # 'B3 (Prompt)': 'results_b3_prompt.json',
}

In [ ]:
def infer_archetype(task_id: str) -> str:
    m = re.match(r'task(\d+)_', str(task_id))
    if not m:
        return 'Unknown'
    n = int(m.group(1))
    if 1 <= n <= 10:
        return 'Sequential Hold'
    if 11 <= n <= 20:
        return 'Selector Search+Hold'
    if 21 <= n <= 30:
        return 'Pick+Place'
    if 31 <= n <= 40:
        return 'Selector Search+Place'
    if 41 <= n <= 50:
        return 'Selector Search+Return'
    return 'Unknown'

def rate(series: pd.Series):
    s = series.dropna()
    if len(s) == 0:
        return pd.NA
    return round(float(s.mean() * 100), 1)

def summarize_block(block: pd.DataFrame, name: str):
    return {
        'Archetype': name,
        'n': int(len(block)),
        'Valid': rate(block['valid']),
        'Grounded': rate(block['grounded']),
        'Params': rate(block['params_valid']),
        'StructComp': rate(block['struct_comp']),
        'Success': rate(block['success']),
    }

In [ ]:
# Load all baselines
baseline_dfs = {}
for baseline_name, results_file in BASELINES.items():
    results_path = Path.cwd() / results_file
    if not results_path.exists():
        print(f"⚠️  {baseline_name}: {results_file} not found, skipping")
        continue
    
    rows = json.loads(results_path.read_text(encoding='utf-8'))
    df = pd.DataFrame(rows)
    
    # Infer archetypes
    if 'archetype' not in df.columns:
        df['archetype_effective'] = df['id'].map(infer_archetype)
    else:
        df['archetype_effective'] = df['archetype'].fillna('')
        mask = df['archetype_effective'].isin(['', 'unlabeled'])
        df.loc[mask, 'archetype_effective'] = df.loc[mask, 'id'].map(infer_archetype)
    
    # Filter out missing BTs
    if 'missing_bt' in df.columns:
        df = df[~df['missing_bt'].fillna(False)]
    
    # Convert metrics to boolean
    metric_cols = ['valid', 'grounded', 'params_valid', 'struct_comp', 'success']
    for c in metric_cols:
        if c in df.columns:
            df[c] = df[c].astype('boolean')
    
    baseline_dfs[baseline_name] = df
    print(f"✓ {baseline_name}: {len(df)} tasks loaded")

In [ ]:
# Summarize each baseline
order = [
    'Sequential Hold',
    'Selector Search+Hold',
    'Pick+Place',
    'Selector Search+Place',
    'Selector Search+Return',
]

summaries = {}
for baseline_name, df in baseline_dfs.items():
    rows_out = [summarize_block(df, 'All tasks')]
    for archetype in order:
        block = df[df['archetype_effective'] == archetype]
        rows_out.append(summarize_block(block, archetype))
    summaries[baseline_name] = pd.DataFrame(rows_out)

# Display each summary
for baseline_name, summary in summaries.items():
    print(f"\n{'='*60}")
    print(f"{baseline_name}")
    print(f"{'='*60}")
    display(summary)

## Table II Format: Main Comparison by Task Archetype

This table matches the format in `experiments_updated.tex` where each metric column shows values for all methods stacked vertically.

In [ ]:
# Create Table II format: each cell contains all baseline values stacked
def format_metric_cell(archetype, metric, summaries, baseline_order):
    """Format a metric cell with all baseline values stacked vertically."""
    values = []
    for baseline in baseline_order:
        if baseline not in summaries:
            values.append('--')
            continue
        summary = summaries[baseline]
        row = summary[summary['Archetype'] == archetype]
        if len(row) == 0:
            values.append('--')
        else:
            val = row.iloc[0][metric]
            values.append(str(val) if pd.notna(val) else '--')
    return '\n'.join(values)

# Define baseline order (M-Core, B1, B2, B3)
baseline_order = ['M-Core', 'B1 (Schema)', 'B2 (Sequence)', 'B3 (Prompt)']

# Create table data
archetypes_with_n = [
    ('All tasks', 50),
    ('Sequential Hold', 10),
    ('Selector Search+Hold', 10),
    ('Pick+Place', 10),
    ('Selector Search+Place', 10),
    ('Selector Search+Return', 10),
]

table_data = []
for archetype, n in archetypes_with_n:
    row = {
        'Archetype': f"{archetype} (n={n})",
        'Valid': format_metric_cell(archetype, 'Valid', summaries, baseline_order),
        'Grounded': format_metric_cell(archetype, 'Grounded', summaries, baseline_order),
        'Params': format_metric_cell(archetype, 'Params', summaries, baseline_order),
        'StructComp': format_metric_cell(archetype, 'StructComp', summaries, baseline_order),
        'Success': format_metric_cell(archetype, 'Success', summaries, baseline_order),
    }
    table_data.append(row)

table_df = pd.DataFrame(table_data)

print("\n" + "="*80)
print("TABLE II: Main Comparison by Task Archetype (rates, %)")
print("Each metric cell shows: M-Core / B1 / B2 / B3")
print("="*80)
display(table_df)

In [ ]:
# Alternative: Create a styled DataFrame with better formatting
import numpy as np

# Create multi-row cells by repeating archetype for each baseline
expanded_rows = []
for archetype, n in archetypes_with_n:
    for baseline in baseline_order:
        if baseline not in summaries:
            vals = {'Valid': '--', 'Grounded': '--', 'Params': '--', 'StructComp': '--', 'Success': '--'}
        else:
            summary = summaries[baseline]
            row = summary[summary['Archetype'] == archetype]
            if len(row) == 0:
                vals = {'Valid': '--', 'Grounded': '--', 'Params': '--', 'StructComp': '--', 'Success': '--'}
            else:
                vals = {
                    'Valid': row.iloc[0]['Valid'],
                    'Grounded': row.iloc[0]['Grounded'],
                    'Params': row.iloc[0]['Params'],
                    'StructComp': row.iloc[0]['StructComp'],
                    'Success': row.iloc[0]['Success'],
                }
        
        expanded_rows.append({
            'Archetype': f"{archetype} (n={n})",
            'Method': baseline,
            **vals
        })

expanded_df = pd.DataFrame(expanded_rows)

# Display with styling
print("\n" + "="*80)
print("TABLE II: Main Comparison (Expanded View)")
print("="*80)

# Style the dataframe
def highlight_archetype(row):
    if row['Method'] == 'M-Core':
        return ['background-color: #f0f0f0'] * len(row)
    elif row['Method'] == 'B1 (Schema)':
        return ['background-color: #e8e8e8'] * len(row)
    elif row['Method'] == 'B2 (Sequence)':
        return ['background-color: #e0e0e0'] * len(row)
    else:
        return ['background-color: #d8d8d8'] * len(row)

styled = expanded_df.style.apply(highlight_archetype, axis=1)
display(styled)

In [ ]:
# Side-by-side comparison for "All tasks"
comparison_all = pd.DataFrame({
    baseline: summary[summary['Archetype'] == 'All tasks'].iloc[0]
    for baseline, summary in summaries.items()
}).T

print("\nAll Tasks Comparison:")
display(comparison_all[['Valid', 'Grounded', 'Params', 'StructComp', 'Success']])

In [ ]:
# Comparison by archetype (Success rate)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

metrics = ['Valid', 'Grounded', 'Params', 'StructComp', 'Success']
for idx, metric in enumerate(metrics):
    ax = axes[idx]
    
    # Prepare data for plotting
    plot_data = {}
    for baseline, summary in summaries.items():
        plot_data[baseline] = summary.set_index('Archetype')[metric]
    
    df_plot = pd.DataFrame(plot_data)
    df_plot = df_plot.loc[order]  # Reorder by archetype
    
    df_plot.plot(kind='bar', ax=ax, rot=45)
    ax.set_title(f'{metric} Rate by Archetype')
    ax.set_ylabel('Rate (%)')
    ax.set_xlabel('')
    ax.legend(title='Baseline', fontsize=8)
    ax.set_ylim(0, 105)

# Remove extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.show()